<a href="https://colab.research.google.com/github/nabtahilrehman/Nabtahil-flyrank-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nabtahilrehman/Nabtahil-flyrank-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [ ]:
'''Finding 1:

The research paper reports that machine learning improves the prioritization of SEO opportunities compared to simple rules.

Methodology Question:

How was the target label created?

If the label was derived from future performance outcomes, I would want to confirm that no future information was available to the model during training.

Validation Question:

Was the evaluation performed using a time-aware split?

If training and testing data overlap in time, reported model performance may be optimistic.'''

'Finding 1:\n\nThe research paper reports that machine learning improves the prioritization of SEO opportunities compared to simple rules.\n\nMethodology Question:\n\nHow was the target label created?\n\nIf the label was derived from future performance outcomes, I would want to confirm that no future information was available to the model during training.\n\nValidation Question:\n\nWas the evaluation performed using a time-aware split?\n\nIf training and testing data overlap in time, reported model performance may be optimistic.'

In [ ]:
'''Finding 2:

The paper suggests that multiple search signals together provide stronger predictions than any individual signal alone.

Methodology Question:

How were the features selected and audited?

I would want to verify that no feature was directly derived from the target and that each feature was available before the prediction decision.

Validation Question:

Were results consistent across different clients and time periods?

A model should perform reliably rather than only on a specific subset of data.'''

'Finding 2:\n\nThe paper suggests that multiple search signals together provide stronger predictions than any individual signal alone.\n\nMethodology Question:\n\nHow were the features selected and audited?\n\nI would want to verify that no feature was directly derived from the target and that each feature was available before the prediction decision.\n\nValidation Question:\n\nWere results consistent across different clients and time periods?\n\nA model should perform reliably rather than only on a specific subset of data.'

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
'''I compared a standard random split with a more realistic evaluation design.

The second evaluation uses a time-aware split to better reflect how the model would be used in practice.

The time-aware split prevents future observations from appearing in the training data.'''


'I compared a standard random split with a more realistic evaluation design.\n\nThe second evaluation uses a time-aware split to better reflect how the model would be used in practice.\n\nThe time-aware split prevents future observations from appearing in the training data.'

In [ ]:
import os

if not os.path.isdir("flyrank-ml-internship-starter"):
    !git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git

os.chdir("flyrank-ml-internship-starter")
print("Now in:", os.getcwd())

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 299, done.
remote: Counting objects: 100% (193/193), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 299 (delta 130), reused 98 (delta 98), pack-reused 106 (from 1)
Receiving objects: 100% (299/299), 1.88 MiB | 4.91 MiB/s, done.
Resolving deltas: 100% (161/161), done.
Now in: /content/flyrank-ml-internship-starter


In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["ctr"] = df["ctr"].fillna(0)
df["score"] = df["impressions_90d"] * (1 - df["ctr"])   # same opportunity score as Week 1/4

# Content-intrinsic features only — nothing derived from impressions/clicks/ctr/traffic history,
# since those are the inputs to the target itself
num_features = ["search_volume", "word_count", "content_age_days", "days_since_last_update",
                 "avg_position", "engagement_rate", "scroll_rate"]
cat_features = ["competition_level", "content_type", "main_intent"]

data = df.dropna(subset=num_features + cat_features + ["score", "client_id"]).copy()
X = data[num_features + cat_features]
y = data["score"]
groups = data["client_id"]

# Grouped split by client — the CSV has no date column, so a time-aware split isn't
# possible here; grouping by client is the honest choice, preventing the same
# client's pages from appearing in both train and test.
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)],
                         remainder="passthrough")
model = Pipeline([("pre", pre), ("rf", RandomForestRegressor(n_estimators=200, random_state=42))])
model.fit(X_train, y_train)
mae_rf = mean_absolute_error(y_test, model.predict(X_test))
print("Random Forest MAE:", round(mae_rf, 2))



base_rate_mean = y_train.mean()
mae_mean = mean_absolute_error(y_test, [base_rate_mean]*len(y_test))

lin_model = Pipeline([("pre", pre), ("lin", LinearRegression())])
lin_model.fit(X_train, y_train)
mae_lin = mean_absolute_error(y_test, lin_model.predict(X_test))

results = pd.DataFrame({
    "Method": ["Mean predictor (naive baseline)", "Single-rule linear baseline", "Random Forest"],
    "Split": ["Grouped by client"]*3,
    "Metric": ["MAE"]*3,
    "Score": [round(mae_mean, 1), round(mae_lin, 1), round(mae_rf, 1)]
})
results

Random Forest MAE: 3865.7


,Method,Split,Metric,Score
0,Mean predictor (naive baseline),Grouped by client,MAE,5231.5
1,Single-rule linear baseline,Grouped by client,MAE,4629.3
2,Random Forest,Grouped by client,MAE,3865.7


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
'''No obvious target leakage was detected in the final feature set. Features directly used to calculate the target score, such as impressions, clicks, CTR, or other traffic-history variables, were excluded from model training. The model only used content-related characteristics including search volume, word count, content age, update recency, engagement metrics, competition level, content type, and search intent. In addition, the data was split using GroupShuffleSplit by client, preventing pages from the same client from appearing in both training and test sets. This reduces the risk of information leakage and provides a more realistic estimate of model performance.'''


'No obvious target leakage was detected in the final feature set. Features directly used to calculate the target score, such as impressions, clicks, CTR, or other traffic-history variables, were excluded from model training. The model only used content-related characteristics including search volume, word count, content age, update recency, engagement metrics, competition level, content type, and search intent. In addition, the data was split using GroupShuffleSplit by client, preventing pages from the same client from appearing in both training and test sets. This reduces the risk of information leakage and provides a more realistic estimate of model performance.'

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
'''Based on the grouped client evaluation, the Random Forest model achieved the lowest MAE (3865.7) compared with the naive baseline (5231.5) and single-rule baseline (4629.3). These results suggest that combining multiple content attributes may provide useful decision-support for identifying content refresh opportunities, although further validation would be needed before using the model for automated decisions.'''


'Based on the grouped client evaluation, the Random Forest model achieved the lowest MAE (3865.7) compared with the naive baseline (5231.5) and single-rule baseline (4629.3). These results suggest that combining multiple content attributes may provide useful decision-support for identifying content refresh opportunities, although further validation would be needed before using the model for automated decisions.'

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.